In [3]:
import example_loader as el
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml
import numpy as np
import jsplib_loader as jl
import plot_utils as pu
import pandas as pd

In [4]:
model = gp.Model()
x = model.addVar(name='x', vtype=gp.GRB.INTEGER)
y = model.addVar(name='y', vtype=gp.GRB.INTEGER)
model.setObjective(x+y, gp.GRB.MAXIMIZE)

model.addConstr(x + 5*y <= 14)
model.addConstr(5*x + y <= 14)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2586148
Academic license 2586148 - for non-commercial use only - registered to br___@vt.edu


<gurobi.Constr *Awaiting Model Update*>

In [6]:
model.params.Presolve = 0
model.params.Method = 1
int_vars, int_idxj = gu.relax_int_or_bin_to_continuous(model)
model.optimize()

Set parameter Presolve to value 0
Set parameter Method to value 1
   Relaxed 0 variables on 
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (linux64 - "EndeavourOS")

CPU model: Intel(R) Core(TM) i7-8750H CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
Method  1
Presolve  0

Academic license 2586148 - for non-commercial use only - registered to br___@vt.edu
Optimize a model with 2 rows, 2 columns and 4 nonzeros
Coefficient statistics:
  Matrix range     [1e+00, 5e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+01]

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  4.666666667e+00


In [4]:
# get score at nearest integer:
# nearest = gu.nearest_integer(int_vars)
# jl.score(int_vars, int_idxj, nearest)

In [7]:
pd.options.display.max_columns = None
pd.options.display.max_rows = None

variables = model.getVars()
cons = model.getConstrs()
def getVarName(idx):
    if idx < len(variables):
        return variables[idx].VarName
    return f"S{idx-len(variables)}"

def getVarValueByName(name):
    rn = model.getVarByName(name)
    if rn:
        return rn.X
    assert name[0] == 'S'
    return cons[int(name[1:])].Slack

basis = gu.read_basis(model)
tableau, col_to_var, negated_rows = gu.read_tableau(model, basis, remove_basis_cols=True)
column_names = [getVarName(c) for c in col_to_var]
df = pd.DataFrame(tableau, columns=column_names)
row_names = [getVarName(c) for c in basis]
row_vals = [getVarValueByName(rn) for rn in row_names]
df.insert(0, "Values", row_vals)
df.insert(0, "Basis", row_names)
df

,Basis,Values,S0,S1
0,y,2.333333,0.208333,-0.041667
1,x,2.333333,-0.041667,0.208333


In [ ]:
# build the transform:
U = np.identity(df.shape[1] - 2)
for j, b in enumerate(basis):
    for i in range(2, df.shape[1]):
        if df.iloc[j, i] < 0:
            U[j, i-2] = -1

,Basis,Values,S0,S1


In [ ]:
def lll_reduction(basis, delta=0.75):
    """Performs the LLL reduction on a lattice basis.

    Args:
        basis (np.ndarray): Input basis vectors as rows in a matrix.
        delta (float): Lovász condition parameter, typically between 0.5 and 1.

    Returns:
        np.ndarray: LLL-reduced basis.
    """
    n = basis.shape[0]
    Q, R = np.linalg.qr(basis.T)
    orthogonal = Q.T
    mu = np.zeros((n, n))

    for i in range(n):
        for j in range(i):
            mu[i, j] = R[j, i] / R[j, j]

    k = 1
    while k < n:
        # Step 1: Size reduction
        for j in range(k - 1, -1, -1):
            q = round(mu[k, j])
            if q != 0:
                basis[k] -= q * basis[j]
                Q, R = np.linalg.qr(basis.T)
                orthogonal = Q.T
                for i in range(n):
                    for j in range(i):
                        mu[i, j] = R[j, i] / R[j, j]

        # Step 2: Lovász condition
        if np.dot(orthogonal[k], orthogonal[k]) >= (delta - mu[k, k - 1]**2) * np.dot(orthogonal[k - 1], orthogonal[k - 1]):
            k += 1
        else:
            # Swap basis vectors
            basis[[k, k - 1]] = basis[[k - 1, k]]
            Q, R = np.linalg.qr(basis.T)
            orthogonal = Q.T
            for i in range(n):
                for j in range(i):
                    mu[i, j] = R[j, i] / R[j, j]
            k = max(k - 1, 1)

    return basis
